<a href="https://colab.research.google.com/github/Fauzi455/PembelajaranMesin_2026/blob/main/JS-06/TUGAS/Tugas%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab


1. Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan,

    a. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.

        i. Gunakan model dengan kernel linier.

        ii. Gunakan model dengan kernel polynomial.

        iii. Gunakan model dengan kernel RBF.

    b. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
from google.colab import files
uploaded = files.upload()

Saving voice.csv to voice.csv


In [5]:
#load data
df = pd.read_csv('voice.csv')

#pisahkan fitur (X) dan label (y)
X = df.drop('label', axis=1)
y = df['label']

#encode label ('male'/'female' menjadi 1/0)
le = LabelEncoder()
y = le.fit_transform(y)

#standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#skenario split dan kernel
split_ratios = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']
results = []

#looping untuk setiap rasio split dan kernel
for split_name, test_size in split_ratios.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    for k in kernels:
        model = SVC(kernel=k)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append({
            'Rasio Split': split_name,
            'Kernel': k,
            'Akurasi': round(acc * 100, 2)
        })

#tabulasi Hasil
df_results = pd.DataFrame(results)
print("Tabel Performansi Model SVM")
display(df_results)

Tabel Performansi Model SVM


,Rasio Split,Kernel,Akurasi
0,70:30,linear,97.06
1,70:30,poly,95.69
2,70:30,rbf,98.11
3,80:20,linear,97.63
4,80:20,poly,96.85
5,80:20,rbf,98.26


2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
train_dir = "/content/drive/MyDrive/images/training/"
test_dir = "/content/drive/MyDrive/images/test/"

In [15]:
import cv2, os, glob
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

train_dir = "/content/drive/MyDrive/images/training/"

X, y = [], []

for label in ['day', 'night']:
    for file in glob.glob(os.path.join(train_dir, label, '*.*')):
        img = cv2.imread(file)
        if img is not None:
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            hist = cv2.calcHist([hsv], [0,1,2], None, [8,8,8], [0,256, 0,256, 0,256])
            X.append(cv2.normalize(hist, hist).flatten())
            y.append(1 if label == 'day' else 0)

#split 80:20 dan scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#hyperparameter tuning & training
param_grid = {'C': [0.1, 1, 10], 'gamma': [1, 0.1, 0.01]}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=3)
grid.fit(X_train_scaled, y_train)

#evaluasi
akurasi = accuracy_score(y_test, grid.predict(X_test_scaled))
print("HASIL KLASIFIKASI")
print(f"Parameter Terbaik : {grid.best_params_}")
print(f"Akurasi Pengujian : {akurasi * 100:.2f}%")

HASIL KLASIFIKASI
Parameter Terbaik : {'C': 10, 'gamma': 0.01}
Akurasi Pengujian : 97.92%
